In [ ]:
# Instalación de dependencias geoespaciales en el entorno de Colab
!pip install pystac-client stackstac rioxarray geopandas rasterio pystac
!pip install odc-stac  #para el load()

In [ ]:
import os
import geopandas as gpd
from pystac_client import Client
import stackstac
import pandas as pd
import rioxarray
import numpy as np
from odc.stac import load
import xarray as xr
import glob
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from google.colab import drive;
# Esto fuerza al sistema a sincronizar los cambios con Drive
os.sync()
print("Sincronización forzada completada.")
from google.colab import drive
drive.mount('/content/drive')
from collections import Counter


In [ ]:
# CONVIERTE DE TIF A PNG
def tif_to_png_batch(input_dir, output_dir, cmap_name='viridis', percentiles=(2, 98)):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 Carpeta creada: {output_dir}")

    # Listar archivos .tif
    files = glob.glob(os.path.join(input_dir, "*.tif"))
    print(f"🚀 Procesando {len(files)} archivos...")

    for tif_path in files:
        filename = os.path.basename(tif_path).replace('.tif', '.png')
        save_path = os.path.join(output_dir, filename)
        with rasterio.open(tif_path) as src:
            data = src.read(1).astype(np.float32)# Leer la 1ra banda (suposición de índice escalar)
            nodata = src.nodata # Reemplazar valores NoData por NaN para el cálculo de estadísticas
            if nodata is not None:
                data[data == nodata] = np.nan

        # --- Optimización de Contraste ---
        vmin, vmax = np.nanpercentile(data, percentiles)# El uso de percentiles evita que píxeles erróneos saturen la imagen
        norm = Normalize(vmin=vmin, vmax=vmax, clip=True)# Normalización matemática al rango [0, 1] para Matplotlib

        # Generar la imagen con la paleta seleccionada
        plt.figure(figsize=(10, 10))
        plt.imshow(data, cmap=cmap_name, norm=norm)
        plt.axis('off')  # Eliminar ejes para presentación académica
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)# Guardar sin márgenes blancos
        plt.close() # Liberar memoria RAM del backend de pyplot
        print(f"✅ Convertido: {filename} [Paleta: {cmap_name}]")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# [MODIFICADO] Cargar shapefile del usuario como ROI
# Original Aedes: bbox_jujuy hardcodeado
# ════════════════════════════════════════════════════════════════════════════

# Ruta del shapefile (ya configurada)
SHAPEFILE_PATH = "/content/drive/MyDrive/Tesis/GEE Murcott/ROI/Roigeneral.zip"

print("🔍 Cargando shapefile para calcular ROI...")
gdf = gpd.read_file(SHAPEFILE_PATH)

# Asegurar CRS en WGS84
if gdf.crs and gdf.crs.is_geographic:
    gdf_geo = gdf
else:
    gdf_geo = gdf.to_crs("EPSG:4326")

# Calcular bounding box [lon_min, lat_min, lon_max, lat_max]
bbox = gdf_geo.total_bounds.tolist()
print(f"✅ Bounding box calculado: {bbox}")

# ─── A partir de aquí, código IDÉNTICO al de Aedes ────────────────────────

# 2. Buscar en el catálogo de AWS
# sentinel-2-l2a (Reflectancia superficial (corregida atmosféricamente),
# sentinel-2-l1c — Reflectancia en tope de atmósfera
# sentinel-2-c1-l2a — Colección 1 L2A (más reciente en Earth Search)
print("🔍 Buscando imágenes Sentinel-2 L2A...")
catalog = Client.open("https://earth-search.aws.element84.com/v1")
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-01-01/2025-12-31",
    query={"eo:cloud_cover": {"lt": 10}} # Máximo 10% de nubes
)
items = search.item_collection()
print(f"✅ Se encontraron {len(items)} escenas.")

# 3. Contar imágenes por mes
conteo = Counter(item.datetime.strftime("%Y-%m") for item in items)

# 4. Crear carpetas si no existen
BASE_DIR = "/content/drive/MyDrive/Mandarina_Murcott/Comparacion_Aedes"
diario_dir = os.path.join(BASE_DIR, "Reporte_STAC")

if not os.path.exists(diario_dir):
    os.makedirs(diario_dir)
    print(f"📁 Carpeta creada: {diario_dir}")

# 5. Guardar reporte
reporte_path = os.path.join(diario_dir, "reporte_cant_imgs_x_mes.txt")

with open(reporte_path, "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("   IMÁGENES DISPONIBLES POR MES — Mandarina Murcott\n")
    f.write("=" * 60 + "\n\n")

    for mes, cantidad in sorted(conteo.items()):
        print(f"  {mes}: {cantidad} imagen{'es' if cantidad > 1 else ''}")
        f.write(f"  {mes}: {cantidad} imagen{'es' if cantidad > 1 else ''}\n")

    f.write(f"\n  Total: {len(items)} escenas en el período.\n")
    f.write("\n" + "=" * 60 + "\n")

print(f"✅ Conteo guardado en: {reporte_path}")

In [ ]:
# ─── Código IDÉNTICO al de Aedes ───────────────────────────────────────────

# 3. CARGA DEL CUBO DE DATOS (Data Cube)
try:
    ds = load(
        items,
        bands=["blue", "green", "red", "nir", "swir16", "swir22"],
        bbox=bbox,
        crs="EPSG:4326",
        resolution=0.0001,
        groupby="solar_day",
        chunks={'time': 1, 'x': 512, 'y': 512}
    )

    print(f"✅ ¡CONEXIÓN EXITOSA!")
    print(f"Dimensiones del área: {dict(ds.sizes)}")

except Exception as e:
    print(f"❌ Error al cargar Data Cube: {e}")